# 5-fold CV: Base vs B vs B'v3 (restart-safe)

Run order in every new Colab session (T4 GPU runtime): **Cell 1, Cell 2, Cell 3**. Cell 4 = analysis, any time.

- Everything is saved to Drive under `videomme_experiments/cv5/`. Re-running Cell 3 skips whatever is already finished, so after a disconnect or the next day you just rerun Cells 1, 2, 3.
- Eval saves after every single generation. Training saves the adapter to Drive when finished (a mid-training disconnect costs at most ~14 min of that one run).
- Fold 0 is your existing 10-video test set and reuses your already trained B and B'v3 adapters. Folds 1 to 4 are the other 40 videos.
- Optional: set `SESSION_BUDGET_MIN` in Cell 2 (e.g. 170) so it stops before starting a task it cannot finish.


In [2]:
# =============================================================================
# CELL 1 - SETUP. Run this FIRST in every fresh Colab runtime (after each reset).
# =============================================================================
import os, sys, subprocess, shutil
from pathlib import Path

os.environ["FORCE_QWENVL_VIDEO_READER"] = "decord"   # must be set before qwen_vl_utils is imported

DRIVE_ROOT   = "/content/drive/MyDrive/videomme_experiments"
SNAP_DIR     = f"{DRIVE_ROOT}/repo_snapshot"
REPO_DIR     = "/content/qwen-vl-finetune"
REPO_URL     = "https://github.com/2U1/Qwen-VL-Series-Finetune.git"
PINNED       = "67078206604820beb20bddbe24c94a0b4a4bd6d4"
DATA_ROOT    = "/content/videomme_data/video-mme-logical"
VIDEO_DIR    = f"{DATA_ROOT}/videos/card_relocation_s_easy"
REQ_TF       = "5.17.0"

def sh(cmd, check=True):
    print("  $", cmd if isinstance(cmd, str) else " ".join(cmd))
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True)
    if check and r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:])
        raise RuntimeError(f"command failed: {cmd}")
    return r

# --- 1. Drive ----------------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
assert os.path.isdir(DRIVE_ROOT), f"{DRIVE_ROOT} not found on Drive"

# --- 2. GPU ------------------------------------------------------------------
gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", check=False)
print("GPU:", gpu.stdout.strip() or "NONE  (Runtime > Change runtime type > T4 GPU)")
assert gpu.stdout.strip(), "No GPU attached"

# --- 3. packages -------------------------------------------------------------
sh(f'{sys.executable} -m pip -q install -U accelerate "qwen-vl-utils[decord]" decord huggingface_hub peft scipy')
tf_now = sh([sys.executable, "-c", "import transformers; print(transformers.__version__)"], check=False).stdout.strip()
if tf_now != REQ_TF:
    sh(f'{sys.executable} -m pip -q install "transformers=={REQ_TF}"')
sh(f"{sys.executable} -m pip -q install --no-deps bitsandbytes==0.50.2 ujson trl==1.13.0")
sh(f"{sys.executable} -m pip -q uninstall -y torchao", check=False)

chk = sh([sys.executable, "-c",
          "import torch, transformers, peft, bitsandbytes, trl, decord, qwen_vl_utils;"
          "print(transformers.__version__, peft.__version__, bitsandbytes.__version__, trl.__version__)"])
print("  versions (transformers peft bnb trl):", chk.stdout.strip())
assert chk.stdout.split()[0] == REQ_TF, f"transformers must be {REQ_TF}"
if "transformers" in sys.modules and sys.modules["transformers"].__version__ != REQ_TF:
    raise RuntimeError("This kernel imported a different transformers earlier. "
                       "Runtime > Restart session, then run Cell 1 again.")

# --- 4. videos (HF dataset is one zip: download it, extract only the 50 card videos) ---
def n_videos():
    return len([f for f in os.listdir(VIDEO_DIR) if f.lower().endswith((".mp4", ".webm", ".mov", ".mkv"))]) \
        if os.path.isdir(VIDEO_DIR) else 0

if n_videos() != 50:
    import zipfile
    from huggingface_hub import hf_hub_download
    os.makedirs("/content/videomme", exist_ok=True)
    os.makedirs("/content/videomme_data", exist_ok=True)
    print("  downloading video_mme_logical.zip (a few minutes on a fresh runtime) ...")
    zpath = hf_hub_download(repo_id="marcuskwan/video-mme-logical", filename="video_mme_logical.zip",
                            repo_type="dataset", local_dir="/content/videomme")
    with zipfile.ZipFile(zpath) as z:
        members = [m for m in z.namelist() if "card_relocation_s_easy/" in m and not m.endswith("/")]
        print(f"  extracting {len(members)} files ...")
        for m in members:
            z.extract(m, "/content/videomme_data")
    if n_videos() != 50:        # zip used a different folder prefix: find the real folder and link it
        hits = [p for p in Path("/content/videomme_data").rglob("card_relocation_s_easy") if p.is_dir()]
        assert hits, "card_relocation_s_easy folder not found after extraction"
        os.makedirs(os.path.dirname(VIDEO_DIR), exist_ok=True)
        if not os.path.exists(VIDEO_DIR):
            os.symlink(str(hits[0]), VIDEO_DIR)
assert n_videos() == 50, f"expected 50 videos, found {n_videos()}"
print("  videos: 50 OK")

# --- 5. repo at pinned commit + saved patches from Drive ----------------------
if not os.path.isdir(f"{REPO_DIR}/.git"):
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    sh(["git", "clone", "-q", REPO_URL, REPO_DIR])
sh(["git", "-C", REPO_DIR, "fetch", "-q", "origin"])
sh(["git", "-C", REPO_DIR, "checkout", "-q", "-f", PINNED])
head = sh(["git", "-C", REPO_DIR, "rev-parse", "HEAD"]).stdout.strip()
assert head == PINNED, f"repo at {head}"

PATCHES = {
    "data_utils.py":          f"{REPO_DIR}/src/dataset/data_utils.py",
    "sft_dataset.py":         f"{REPO_DIR}/src/dataset/sft_dataset.py",
    "monkey_patch_vision.py": f"{REPO_DIR}/src/train/monkey_patch_vision.py",
}
for name in PATCHES:
    if not os.path.isfile(f"{SNAP_DIR}/{name}"):
        raise FileNotFoundError(f"{SNAP_DIR}/{name} missing. Re-run your old Step 3 cell once so it saves "
                                "the patched files to Drive, then come back.")
for name, dst in PATCHES.items():
    Path(dst).write_text(Path(f"{SNAP_DIR}/{name}").read_text())
    compile(Path(dst).read_text(), name, "exec")
assert "nframes=None" in Path(PATCHES["data_utils.py"]).read_text()
assert "nframes=self.nframes" in Path(PATCHES["sft_dataset.py"]).read_text()
print("  repo + patches restored from Drive snapshot")

# --- 6. smoke test of the patched video pipeline ------------------------------
sys.path.insert(0, f"{REPO_DIR}/src")
for m in ("dataset", "dataset.data_utils"):
    sys.modules.pop(m, None)
from dataset import data_utils
vid, _ = data_utils.get_video_info(f"{VIDEO_DIR}/card_relocation_s_L1_0001_v3.mp4", min_pixels=100352,
                                   max_pixels=602112, width=224, height=224, fps=None,
                                   image_patch_size=14, nframes=24, return_video_metadata=False)
assert vid.shape[0] == 24, vid.shape
print("  video pipeline smoke test: PASS", tuple(vid.shape))
print("\nCELL 1 DONE. Now run Cell 2 (definitions), then Cell 3 (pipeline).")

Mounted at /content/drive
  $ nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
GPU: Tesla T4, 15360 MiB
  $ /usr/bin/python3 -m pip -q install -U accelerate "qwen-vl-utils[decord]" decord huggingface_hub peft scipy
  $ /usr/bin/python3 -c import transformers; print(transformers.__version__)
  $ /usr/bin/python3 -m pip -q install "transformers==5.17.0"
  $ /usr/bin/python3 -m pip -q install --no-deps bitsandbytes==0.50.2 ujson trl==1.13.0
  $ /usr/bin/python3 -m pip -q uninstall -y torchao
  $ /usr/bin/python3 -c import torch, transformers, peft, bitsandbytes, trl, decord, qwen_vl_utils;print(transformers.__version__, peft.__version__, bitsandbytes.__version__, trl.__version__)
  versions (transformers peft bnb trl): 5.17.0 0.21.2 0.50.2 1.13.0
  downloading video_mme_logical.zip (a few minutes on a fresh runtime) ...


video_mme_logical.zip: reconstructing file:   0%|          |  0.00B / 3.56GB            

video_mme_logical.zip: downloading bytes:           |  0.00B            

  extracting 50 files ...
  videos: 50 OK
  $ git clone -q https://github.com/2U1/Qwen-VL-Series-Finetune.git /content/qwen-vl-finetune
  $ git -C /content/qwen-vl-finetune fetch -q origin
  $ git -C /content/qwen-vl-finetune checkout -q -f 67078206604820beb20bddbe24c94a0b4a4bd6d4
  $ git -C /content/qwen-vl-finetune rev-parse HEAD
  repo + patches restored from Drive snapshot


qwen-vl-utils using decord to read video.


  video pipeline smoke test: PASS (24, 3, 224, 224)

CELL 1 DONE. Now run Cell 2 (definitions), then Cell 3 (pipeline).


In [3]:
# =============================================================================
# CELL 2 - DEFINITIONS (config + all functions). No GPU work happens here.
# Run after Cell 1, every session.
# =============================================================================
import os, sys, re, json, gc, time, math, random, shutil, subprocess, traceback
from collections import Counter
from datetime import datetime
from pathlib import Path

os.environ["FORCE_QWENVL_VIDEO_READER"] = "decord"

# ----------------------------- CONFIG ----------------------------------------
DRIVE_ROOT    = "/content/drive/MyDrive/videomme_experiments"
SFT_DIR       = f"{DRIVE_ROOT}/sft_intervention"
GT_PATH       = f"{DRIVE_ROOT}/ground_truth/card_relocation_L1_0001_0050_raw_manifest_records.json"
SPLIT_SUMMARY = f"{SFT_DIR}/split_summary.json"
CV_ROOT       = f"{DRIVE_ROOT}/cv5"                       # everything new lives here
VIDEO_DIR     = "/content/videomme_data/video-mme-logical/videos/card_relocation_s_easy"
REPO_DIR      = "/content/qwen-vl-finetune"
LOCAL_TMP     = "/content/cv_tmp"

MODEL_ID      = "Qwen/Qwen2.5-VL-3B-Instruct"
REQ_TF        = "5.17.0"

N_FOLDS       = 5
CV_SEED       = 20261001            # only used to deal the 40 non-test videos into folds 1..4
CORRUPTION_SEED_V3 = 56182047       # same base seed as your B' v3 (per fold: + fold index, fallbacks only)
POSITIONS     = ["A", "B", "C", "D"]
INF_COST      = 10 ** 6
MAX_DERANGE_TRIES = 500

CONDITIONS    = ["B", "Bprime_v3"]  # trained conditions
VARIANTS      = ["final", "trajectory"]            # add "transition" only if you also want it (slower)
MAX_NEW       = {"final": 16, "trajectory": 256}
NFRAMES, RESIZED = 24, 224
SEED          = 20260917

REUSE_FOLD0   = True                # fold 0 = your existing 40/10 split; reuse its trained adapters
RUN_BASE      = True                # evaluate the untouched base model on all 50 videos (4-bit, same pipeline)

# Optional: stop cleanly BEFORE starting a task that would not fit. e.g. 170 for a ~3h session. None = no limit.
SESSION_BUDGET_MIN = None
EST_MIN = {"train": 15, "eval": 6, "base": 18}     # rough minutes per task on a T4 (from your logs)

TRAIN_ARGS = {
    "--model_id": MODEL_ID, "--image_folder": VIDEO_DIR,
    "--lora_enable": "True", "--lora_rank": "16", "--lora_alpha": "32", "--lora_dropout": "0.05",
    "--num_lora_modules": "-1", "--freeze_llm": "True", "--freeze_vision_tower": "True",
    "--freeze_merger": "True", "--bits": "4", "--quant_type": "nf4", "--double_quant": "True",
    "--use_liger_kernel": "False", "--disable_flash_attn2": "True", "--fp16": "True", "--bf16": "False",
    "--gradient_checkpointing": "True", "--per_device_train_batch_size": "1",
    "--gradient_accumulation_steps": "8", "--num_train_epochs": "3", "--learning_rate": "0.0001",
    "--weight_decay": "0.01", "--lr_scheduler_type": "cosine", "--optim": "adamw_bnb_8bit",
    "--max_seq_length": "4096", "--video_resized_width": "224", "--video_resized_height": "224",
    "--nframes": "24", "--remove_unused_columns": "False", "--lazy_preprocess": "True",
    "--dataloader_num_workers": "0", "--logging_steps": "1", "--save_strategy": "no", "--report_to": "none",
}
TRAIN_TIMEOUT_SEC = 3600

FOLD0_EXISTING = {"B": f"{SFT_DIR}/models/B_trajectory", "Bprime_v3": f"{SFT_DIR}/models/Bprime_v3"}

# ----------------------------- small helpers ---------------------------------
def jdump(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

def nid(s):
    m = re.search(r"_(\d{4})_", s)
    return int(m.group(1)) if m else 0

def log(msg=""):
    print(msg, flush=True)

# ----------------------------- ground truth + folds --------------------------
def load_meta():
    with open(GT_PATH) as f:
        gt = json.load(f)
    assert len(gt) == 50, f"expected 50 GT records, got {len(gt)}"
    rows = []
    for vid, item in gt.items():
        ans = item["answer"]
        ans = json.loads(ans) if isinstance(ans, str) else ans
        trace = ans["position_trace"]
        keys = sorted(trace.keys(), key=int)
        pos = [trace[k] for k in keys]
        init = ans["initial_position"]
        moves, prev = [], init
        for p in pos:
            moves.append(0 if p == prev else 1)
            prev = p
        rows.append({
            "id": vid, "video_file": os.path.basename(item["video_path"]),
            "target_card": ans["target_card"], "initial_position": init,
            "round_positions": pos, "final_position": pos[-1], "move_trace": moves,
        })
    rows.sort(key=lambda r: nid(r["id"]))
    return rows

def build_folds(meta):
    """Fold 0 = your existing 10 test videos. Folds 1-4 = the other 40, stratified by final position.
    Written once to Drive; on every later run it is LOADED, never regenerated."""
    path = f"{CV_ROOT}/cv_folds.json"
    if os.path.isfile(path):
        folds = json.load(open(path))["folds"]
    else:
        fold0 = list(json.load(open(SPLIT_SUMMARY))["test_ids"])
        rest = [r for r in meta if r["id"] not in set(fold0)]
        rng = random.Random(CV_SEED)
        ordered = []
        for pos in POSITIONS:
            g = sorted([r["id"] for r in rest if r["final_position"] == pos], key=nid)
            rng.shuffle(g)
            ordered.extend(g)
        folds = [sorted(fold0, key=nid)] + [[] for _ in range(N_FOLDS - 1)]
        for i, v in enumerate(ordered):
            folds[1 + i % (N_FOLDS - 1)].append(v)
        folds = [sorted(f, key=nid) for f in folds]
        jdump(path, {"seed": CV_SEED, "note": "fold 0 = original fixed test set", "folds": folds})
    ids = [r["id"] for r in meta]
    flat = [v for f in folds for v in f]
    assert sorted(flat) == sorted(ids), "folds do not partition the 50 videos"
    assert all(len(f) == 10 for f in folds), [len(f) for f in folds]
    return folds

_CACHE = {}
def world():
    if "meta" not in _CACHE:
        meta = load_meta()
        folds = build_folds(meta)
        _CACHE.update(meta=meta, folds=folds, by_id={r["id"]: r for r in meta},
                      fold_of={v: k for k, f in enumerate(folds) for v in f})
    return _CACHE

def train_rows_for(k):
    w = world()
    test = set(w["folds"][k])
    return [r for r in w["meta"] if r["id"] not in test]

# ----------------------------- prompts + training data ------------------------
def prompt_final(card):
    return f"""Watch the full card trick video.

The target card is the card labeled {card}.

There are four fixed screen positions:
A, B, C, and D.

Track the target card through the entire video.

What is the final position of the target card?

Answer with ONLY one letter:
A
B
C
or
D"""

def prompt_trajectory(card):
    return f"""Watch the full card trick video.

The target card is the card labeled {card}.

There are four fixed screen positions:
A, B, C, and D.

Track the target card through every round.

Report the target card's position after each round,
in chronological order.

Use exactly this format:

Initial: X
Round 1: X
Round 2: X
...
Final: X

Replace X with A, B, C, or D.

Do not add anything else."""

PROMPTS = {"final": prompt_final, "trajectory": prompt_trajectory}

def make_record(row, prompt, answer):
    return {"id": row["id"], "video": row["video_file"],
            "conversations": [{"from": "human", "value": "<video>\n" + prompt},
                              {"from": "gpt", "value": answer}]}

def render_traj(initial, rounds, final):
    lines = [f"Initial: {initial}"] + [f"Round {i}: {p}" for i, p in enumerate(rounds, 1)] + [f"Final: {final}"]
    return "\n".join(lines)

def move_count_of(initial, seq):
    prev, n = initial, 0
    for p in seq:
        n += int(p != prev)
        prev = p
    return n

def build_B(train_rows):
    return [make_record(r, prompt_trajectory(r["target_card"]),
                        render_traj(r["initial_position"], r["round_positions"], r["final_position"]))
            for r in train_rows]

def build_Bprime_v3(train_rows, fold):
    """Same algorithm as your Step 5v3: move-count-matched cross-example trajectories via Hungarian."""
    import numpy as np
    from scipy.optimize import linear_sum_assignment
    rng = random.Random(CORRUPTION_SEED_V3 + fold)

    def derange(seq):
        n = len(seq)
        if n <= 1:
            return None
        idx = list(range(n))
        for _ in range(MAX_DERANGE_TRIES):
            rng.shuffle(idx)
            if all(seq[idx[i]] != seq[i] for i in range(n)):
                return [seq[idx[i]] for i in range(n)]
        return None

    def substitute(seq):
        return [rng.choice([p for p in POSITIONS if p != v]) for v in seq]

    groups = {}
    for r in train_rows:
        n_int = len(r["round_positions"]) - 1
        if n_int <= 0:
            raise RuntimeError(f"{r['id']}: no intermediate rounds, cannot corrupt")
        groups.setdefault(n_int, []).append(r)

    records, audit, methods = [], [], Counter()
    for n_int, group in sorted(groups.items()):
        col, valid = None, None
        if len(group) >= 2:
            n = len(group)
            true_seqs = [g["round_positions"][:-1] for g in group]
            inits = [g["initial_position"] for g in group]
            finals = [g["final_position"] for g in group]
            true_mv = [move_count_of(inits[i], group[i]["round_positions"]) for i in range(n)]
            cost = np.zeros((n, n))
            for i in range(n):
                for j in range(n):
                    if i == j or true_seqs[j] == true_seqs[i]:
                        cost[i, j] = INF_COST
                    else:
                        cost[i, j] = abs(true_mv[i] - move_count_of(inits[i], true_seqs[j] + [finals[i]]))
            _, col = linear_sum_assignment(cost)
            valid = [cost[i, col[i]] < INF_COST for i in range(n)]
        for slot, row in enumerate(group):
            true_seq = row["round_positions"][:-1]
            src, assigned = None, None
            if col is not None and valid[slot]:
                src = group[col[slot]]
                assigned, method = list(src["round_positions"][:-1]), "hungarian_move_matched"
                src = src["id"]
            if assigned is None:
                d = derange(true_seq)
                assigned, method = (d, "intra_derangement") if d is not None else (substitute(true_seq), "fallback_substitution")
            corrupted = assigned + [row["final_position"]]
            methods[method] += 1
            assert len(corrupted) == len(row["round_positions"])
            assert corrupted[-1] == row["final_position"]
            assert corrupted[:-1] != true_seq, f"{row['id']}: corruption identical to GT"
            changed = sum(1 for i in range(n_int) if corrupted[i] != row["round_positions"][i])
            records.append(make_record(row, prompt_trajectory(row["target_card"]),
                                       render_traj(row["initial_position"], corrupted, row["final_position"])))
            audit.append({"id": row["id"], "source": src, "method": method, "n_intermediate": n_int,
                          "n_changed": changed, "true": row["round_positions"], "corrupted": corrupted,
                          "true_moves": move_count_of(row["initial_position"], row["round_positions"]),
                          "corrupted_moves": move_count_of(row["initial_position"], corrupted)})
    order = {r["id"]: i for i, r in enumerate(train_rows)}
    records.sort(key=lambda x: order[x["id"]])
    audit.sort(key=lambda x: order[x["id"]])

    # validation gates (same spirit as Step 5v3)
    by_b = {r["id"]: r for r in build_B(train_rows)}
    for rec in records:
        b = by_b[rec["id"]]
        assert rec["conversations"][0]["value"] == b["conversations"][0]["value"], "prompt differs from B"
        assert rec["conversations"][1]["value"] != b["conversations"][1]["value"], "answer identical to B"
        assert len(rec["conversations"][1]["value"].splitlines()) == len(b["conversations"][1]["value"].splitlines())
    gt_f, co_f = Counter(), Counter()
    for a in audit:
        gt_f.update(a["true"]); co_f.update(a["corrupted"])
    mh_t, mh_c = Counter(a["true_moves"] for a in audit), Counter(a["corrupted_moves"] for a in audit)
    summary = {
        "fold": fold, "n": len(audit), "methods": dict(methods),
        "mean_moves_true": round(sum(a["true_moves"] for a in audit) / len(audit), 3),
        "mean_moves_corrupted": round(sum(a["corrupted_moves"] for a in audit) / len(audit), 3),
        "move_hist_l1": sum(abs(mh_t.get(m, 0) - mh_c.get(m, 0)) for m in set(mh_t) | set(mh_c)),
        "letter_freq_l1": sum(abs(gt_f.get(p, 0) - co_f.get(p, 0)) for p in POSITIONS),
        "majority_diverged": sum(1 for a in audit if a["n_changed"] > a["n_intermediate"] / 2),
    }
    return records, audit, summary

# ----------------------------- paths / done-checks ----------------------------
def adapter_valid(d):
    return os.path.isfile(f"{d}/adapter_config.json") and (
        os.path.isfile(f"{d}/adapter_model.safetensors") or os.path.isfile(f"{d}/adapter_model.bin"))

def adapter_path(cond, k):
    if k == 0 and REUSE_FOLD0 and adapter_valid(FOLD0_EXISTING[cond]):
        return FOLD0_EXISTING[cond]
    return f"{CV_ROOT}/models/fold{k}/{cond}"

def train_done(cond, k):
    d = adapter_path(cond, k)
    if k == 0 and d == FOLD0_EXISTING[cond]:
        return True
    return os.path.isfile(f"{d}/DONE.json") and adapter_valid(d)

def raw_path(cond, k):
    return f"{CV_ROOT}/eval/Base/all/raw_outputs.jsonl" if cond == "Base" else f"{CV_ROOT}/eval/{cond}/fold{k}/raw_outputs.jsonl"

def eval_ids(cond, k):
    w = world()
    return sorted([r["id"] for r in w["meta"]], key=nid) if cond == "Base" else list(w["folds"][k])

def done_keys(path):
    keys = set()
    if os.path.isfile(path):
        for line in open(path):
            line = line.strip()
            if line:
                try:
                    r = json.loads(line)
                    keys.add((r["id"], r["variant"]))
                except Exception:
                    pass
    return keys

def eval_done(cond, k):
    need = {(v, var) for var in VARIANTS for v in eval_ids(cond, k)}
    return need <= done_keys(raw_path(cond, k))

def build_tasks():
    tasks = []
    for k in range(N_FOLDS):
        for c in CONDITIONS:
            tasks.append(("train", c, k))
        for c in CONDITIONS:
            tasks.append(("eval", c, k))
    if RUN_BASE:
        tasks.append(("eval", "Base", None))
    return tasks

def task_done(t):
    kind, cond, k = t
    return train_done(cond, k) if kind == "train" else eval_done(cond, k)

def status():
    tasks = build_tasks()
    log("=" * 70); log("CV STATUS"); log("=" * 70)
    for t in tasks:
        kind, cond, k = t
        name = f"{kind:5s} {cond:10s} fold {k}" if k is not None else f"{kind:5s} {cond:10s} all 50"
        extra = ""
        if kind == "eval":
            n = len({x for x in done_keys(raw_path(cond, k))})
            extra = f"  ({n}/{len(VARIANTS) * len(eval_ids(cond, k))} jobs)"
        log(f"  [{'DONE' if task_done(t) else 'todo'}] {name}{extra}")
    left = [t for t in tasks if not task_done(t)]
    log(f"\n  {len(tasks) - len(left)}/{len(tasks)} tasks done")
    return left

# ----------------------------- training --------------------------------------
def verify_adapter(d):
    from safetensors import safe_open
    assert adapter_valid(d), f"adapter missing in {d}"
    w = f"{d}/adapter_model.safetensors"
    if os.path.isfile(w):
        bad = 0
        with safe_open(w, framework="pt", device="cpu") as f:
            for key in f.keys():
                if not f.get_tensor(key).isfinite().all():
                    bad += 1
        assert bad == 0, f"{bad} non-finite tensors in adapter"

def stream_run(cmd, cwd, env, log_path):
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    t0, beat = time.time(), time.time()
    keys = ("loss", "train_runtime", "Traceback", "Error", "OutOfMemory")
    with open(log_path, "w") as lf:
        p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in p.stdout:
                lf.write(line)
                if any(x in line for x in keys) and len(line) < 600:
                    log("   " + line.rstrip())
                if time.time() - beat > 120:
                    log(f"   ... still training ({int(time.time() - t0)}s)"); beat = time.time()
                if time.time() - t0 > TRAIN_TIMEOUT_SEC:
                    p.kill(); raise TimeoutError("training exceeded timeout")
        finally:
            p.wait()
    return p.returncode, round(time.time() - t0, 1)

def do_train(cond, k):
    rows = train_rows_for(k)
    assert len(rows) == 40
    ddir = f"{CV_ROOT}/data/fold{k}"
    if cond == "B":
        recs = build_B(rows)
    else:
        recs, audit, summ = build_Bprime_v3(rows, k)
        jdump(f"{ddir}/bprime_v3_audit.json", {"summary": summ, "rows": audit})
        log(f"   B'v3 fold {k} audit: {summ}")
    data_path = f"{ddir}/train_{cond}.json"
    jdump(data_path, recs)
    out_final = adapter_path(cond, k)
    shutil.rmtree(out_final, ignore_errors=True)            # drop any half-written leftovers
    tmp_out = f"{LOCAL_TMP}/fold{k}_{cond}"
    shutil.rmtree(tmp_out, ignore_errors=True); os.makedirs(tmp_out)
    args = dict(TRAIN_ARGS); args["--data_path"] = data_path; args["--output_dir"] = tmp_out
    cmd = [sys.executable, "src/train/train_sft.py"] + [x for kv in args.items() for x in kv]
    env = dict(os.environ)
    env["PYTHONPATH"] = f"{REPO_DIR}/src:" + env.get("PYTHONPATH", "")
    env["FORCE_QWENVL_VIDEO_READER"] = "decord"
    lp = f"{CV_ROOT}/logs/train_{cond}_fold{k}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
    rc, el = stream_run(cmd, REPO_DIR, env, lp)
    if rc != 0:
        raise RuntimeError(f"training failed (exit {rc}). Log: {lp}")
    verify_adapter(tmp_out)
    shutil.copytree(tmp_out, out_final)                      # copy to Drive, THEN mark done
    jdump(f"{out_final}/DONE.json", {"cond": cond, "fold": k, "elapsed_sec": el, "log": lp,
                                     "finished": datetime.now().isoformat(timespec="seconds")})
    shutil.rmtree(tmp_out, ignore_errors=True)
    log(f"   trained {cond} fold {k} in {el}s -> {out_final}")

# ----------------------------- parsing + scoring (same logic as Steps 1-7) -----
LETTER = r"[ABCD]"
ENUM_RE = re.compile(r"\bA\s*,\s*B\s*,\s*C\s*,?\s*(?:AND|OR)\s*D\b", re.I)

def _clean(t):
    t = re.sub(r"[*_`#]+", " ", t.strip())
    return ENUM_RE.sub(" ", t)

def parse_output(text):
    out = {"final": None, "final_method": None, "initial": None, "rounds": {}, "round_moves": {}}
    up = _clean(text).upper()
    m = re.search(rf"\bINITIAL(?:\s+POSITION)?\s*[:\-]?\s*\(?({LETTER})\b", up)
    if m:
        out["initial"] = m.group(1)
    for rm in re.finditer(rf"\bROUND\s*(\d+)\s*[:\-]?\s*([^\n]*)", up):
        idx, body = int(rm.group(1)), rm.group(2)
        mv = re.search(rf"MOVED?\s+FROM\s+\(?({LETTER})\)?\s+TO\s+\(?({LETTER})\b", body)
        if mv:
            out["rounds"][idx] = mv.group(2); out["round_moves"][idx] = 1; continue
        st = re.search(rf"STAY(?:ED|S)?\s+(?:AT|IN)\s+\(?({LETTER})\b", body)
        if st:
            out["rounds"][idx] = st.group(1); out["round_moves"][idx] = 0; continue
        bare = re.search(rf"\b({LETTER})\b", body)
        if bare:
            out["rounds"][idx] = bare.group(1)
        if re.search(r"\bMOVED?\b", body):
            out["round_moves"][idx] = 1
        elif re.search(r"\bSTAY", body) or re.search(r"\bDID\s+NOT\s+MOVE\b", body):
            out["round_moves"][idx] = 0
    s = up.strip().strip(".").strip()
    if len(s) == 1 and s in "ABCD":
        out["final"], out["final_method"] = s, "bare_letter"
    else:
        m = re.search(rf"\bFINAL(?:\s+POSITION)?\s*[:\-]?\s*(?:IS\s+)?\(?({LETTER})\b", up)
        if m:
            out["final"], out["final_method"] = m.group(1), "final_label"
    if out["final"] is None:
        m = re.search(rf"\b(?:POSITION|ANSWER|CARD)\s+(?:IS\s+)?\(?({LETTER})\b", up)
        if m:
            out["final"], out["final_method"] = m.group(1), "answer_phrase"
    if out["final"] is None and out["rounds"]:
        out["final"], out["final_method"] = out["rounds"][max(out["rounds"])], "last_round"
    if out["final"] is None:
        letters = re.findall(rf"\b({LETTER})\b", up)
        if letters:
            out["final"], out["final_method"] = letters[-1], "last_letter_fallback"
    return out

# ----------------------------- evaluation -------------------------------------
_PROC = {}
def get_processor():
    if "p" not in _PROC:
        from transformers import AutoProcessor
        _PROC["p"] = AutoProcessor.from_pretrained(MODEL_ID)
    return _PROC["p"]

def build_messages(video_path, prompt):
    return [{"role": "user", "content": [
        {"type": "video", "video": video_path, "nframes": NFRAMES, "resized_height": RESIZED, "resized_width": RESIZED},
        {"type": "text", "text": prompt}]}]

def prepare_inputs(video_path, prompt):
    from qwen_vl_utils import process_vision_info
    proc = get_processor()
    msgs = build_messages(video_path, prompt)
    imgs, vids, vk = process_vision_info(msgs, return_video_kwargs=True)
    if isinstance(vk, dict):
        fps = vk.get("fps")
        if isinstance(fps, (list, tuple)) and len(fps) == 1:
            vk["fps"] = float(fps[0])
    text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return proc(text=[text], images=imgs, videos=vids, padding=True, return_tensors="pt",
                **(vk if isinstance(vk, dict) else {}))

def generate(model, video_path, prompt, max_new):
    import torch
    with torch.inference_mode():
        inp = prepare_inputs(video_path, prompt)
        inp = {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in inp.items()}
        n = inp["input_ids"].shape[1]
        ids = model.generate(**inp, max_new_tokens=max_new, do_sample=False)
        text = get_processor().batch_decode([o[n:] for o in ids], skip_special_tokens=True,
                                            clean_up_tokenization_spaces=False)[0]
    del inp, ids
    gc.collect(); torch.cuda.empty_cache()
    return text

def do_eval(cond, k):
    import torch
    from transformers import Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
    w = world()
    path = raw_path(cond, k)
    done = done_keys(path)
    jobs = [(v, var) for var in VARIANTS for v in eval_ids(cond, k) if (v, var) not in done]
    if not jobs:
        return
    log(f"   {len(jobs)} generation jobs to run ({len(done)} already on Drive)")
    torch.manual_seed(SEED)
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    base = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
    if cond == "Base":
        model = base
    else:
        from peft import PeftModel
        model = PeftModel.from_pretrained(base, adapter_path(cond, k))
    model.eval()
    log(f"   model loaded, VRAM {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    t0, fails = time.time(), 0
    os.makedirs(os.path.dirname(path), exist_ok=True)
    for i, (vid, var) in enumerate(jobs, 1):
        gt = w["by_id"][vid]
        t1 = time.time()
        try:
            text = generate(model, f"{VIDEO_DIR}/{gt['video_file']}", PROMPTS[var](gt["target_card"]), MAX_NEW[var])
        except Exception as e:
            fails += 1
            log(f"   FAILED {vid} {var}: {type(e).__name__}: {str(e)[:150]}  (will retry on next run)")
            gc.collect(); torch.cuda.empty_cache()
            continue
        p = parse_output(text)
        row = {"id": vid, "variant": var, "cond": cond, "fold": w["fold_of"][vid], "raw_output": text,
               "parsed": {"final": p["final"], "initial": p["initial"],
                          "rounds": {str(a): b for a, b in sorted(p["rounds"].items())}},
               "ground_truth": {"initial_position": gt["initial_position"], "round_positions": gt["round_positions"],
                                "final_position": gt["final_position"]},
               "seconds": round(time.time() - t1, 2), "ts": datetime.now().isoformat(timespec="seconds")}
        with open(path, "a") as f:                      # append per job: a disconnect loses at most 1 job
            f.write(json.dumps(row) + "\n")
        ok = "?" if p["final"] is None else ("OK" if p["final"] == gt["final_position"] else "X ")
        if i % 5 == 0 or i == len(jobs):
            log(f"   [{i}/{len(jobs)}] {vid} {var} pred={p['final']} gt={gt['final_position']} {ok}")
    del model, base
    gc.collect(); torch.cuda.empty_cache()
    log(f"   eval wall time {time.time() - t0:.0f}s, failures {fails}")

# ----------------------------- driver -----------------------------------------
def preflight(need_train):
    import torch, transformers
    assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then rerun Cell 1."
    if transformers.__version__ != REQ_TF:
        raise RuntimeError(f"transformers {transformers.__version__} != {REQ_TF}. Run Cell 1, restart session if told to.")
    for p in (GT_PATH, SPLIT_SUMMARY):
        assert os.path.isfile(p), f"missing {p}"
    n = len([f for f in os.listdir(VIDEO_DIR)]) if os.path.isdir(VIDEO_DIR) else 0
    assert n == 50, "videos not restored. Run Cell 1."
    if need_train:
        assert os.path.isfile(f"{REPO_DIR}/src/train/train_sft.py"), "repo missing. Run Cell 1."
        assert "nframes=self.nframes" in Path(f"{REPO_DIR}/src/dataset/sft_dataset.py").read_text(), "patches missing. Run Cell 1."

def run_pipeline():
    t_start = time.time()
    world()
    left = status()
    if not left:
        log("\nEverything is finished. Run Cell 4 for the analysis.")
        return
    preflight(need_train=any(t[0] == "train" for t in left))
    os.makedirs(LOCAL_TMP, exist_ok=True)
    for t in left:
        kind, cond, k = t
        key = "base" if cond == "Base" else kind
        used = (time.time() - t_start) / 60
        if SESSION_BUDGET_MIN and used + EST_MIN[key] > SESSION_BUDGET_MIN:
            log(f"\nBudget guard: {used:.0f} min used, next task needs ~{EST_MIN[key]} min. Stopping cleanly.")
            log("Rerun Cell 3 in a new session to continue.")
            break
        name = f"{kind} {cond} fold {k}" if k is not None else f"{kind} {cond} (all 50)"
        log("\n" + "-" * 70); log(f"{datetime.now().strftime('%H:%M:%S')}  START {name}   ({used:.0f} min into session)"); log("-" * 70)
        t1 = time.time()
        if kind == "train":
            do_train(cond, k)
        else:
            do_eval(cond, k)
        log(f"   finished {name} in {(time.time() - t1) / 60:.1f} min")
        gc.collect()
        try:
            import torch; torch.cuda.empty_cache()
        except Exception:
            pass
    left = status()
    log("\nALL DONE. Run Cell 4." if not left else f"\n{len(left)} task(s) remaining. Rerun Cell 3 (after Cell 1+2 if the runtime reset).")


In [4]:
# =============================================================================
# CELL 3 - RUN. Resumes automatically. Rerun after Cells 1 and 2 in any new session.
# =============================================================================
run_pipeline()


CV STATUS
  [DONE] train B          fold 0
  [DONE] train Bprime_v3  fold 0
  [DONE] eval  B          fold 0  (20/20 jobs)
  [DONE] eval  Bprime_v3  fold 0  (20/20 jobs)
  [DONE] train B          fold 1
  [DONE] train Bprime_v3  fold 1
  [DONE] eval  B          fold 1  (20/20 jobs)
  [DONE] eval  Bprime_v3  fold 1  (20/20 jobs)
  [DONE] train B          fold 2
  [DONE] train Bprime_v3  fold 2
  [DONE] eval  B          fold 2  (20/20 jobs)
  [DONE] eval  Bprime_v3  fold 2  (20/20 jobs)
  [DONE] train B          fold 3
  [DONE] train Bprime_v3  fold 3
  [DONE] eval  B          fold 3  (20/20 jobs)
  [DONE] eval  Bprime_v3  fold 3  (20/20 jobs)
  [DONE] train B          fold 4
  [DONE] train Bprime_v3  fold 4
  [DONE] eval  B          fold 4  (20/20 jobs)
  [DONE] eval  Bprime_v3  fold 4  (20/20 jobs)
  [DONE] eval  Base       all 50  (100/100 jobs)

  21/21 tasks done

Everything is finished. Run Cell 4 for the analysis.


In [5]:
# =============================================================================
# CELL 4 - ANALYSIS. Needs only Drive (no GPU). Safe to run at any time, even half-finished:
# it only uses videos that have BOTH B and B'v3 results. Needs Cell 2 run first.
# =============================================================================
import numpy as np

def load_rows(cond):
    rows = {}
    ks = [None] if cond == "Base" else range(N_FOLDS)
    for k in ks:
        p = raw_path(cond, k)
        if os.path.isfile(p):
            for line in open(p):
                line = line.strip()
                if line:
                    r = json.loads(line)
                    rows[(r["id"], r["variant"])] = r          # last write wins
    return rows

def per_video(r):
    gt, p = r["ground_truth"], r["parsed"]
    rounds = {int(a): b for a, b in p["rounds"].items()}
    n = len(gt["round_positions"])
    rc = sum(1 for i, g in enumerate(gt["round_positions"], 1) if rounds.get(i) == g)
    return {"final": int(p["final"] == gt["final_position"]) if p["final"] else 0,
            "per_round": rc / n, "exact": int(rc == n), "parsed": int(p["final"] is not None)}

def cp_ci(x, n, a=0.05):
    from scipy.stats import beta
    lo = 0.0 if x == 0 else beta.ppf(a / 2, x, n - x + 1)
    hi = 1.0 if x == n else beta.ppf(1 - a / 2, x + 1, n - x)
    return lo, hi

def boot_ci(v, B=10000, seed=1):
    rng = np.random.default_rng(seed)
    v = np.asarray(v, float)
    m = v[rng.integers(0, len(v), (B, len(v)))].mean(1)
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

def signflip_p(d, R=100000, seed=2):
    d = np.asarray(d, float)
    if np.all(d == 0):
        return 1.0
    rng = np.random.default_rng(seed)
    s = rng.choice([-1.0, 1.0], size=(R, len(d)))
    obs = abs(d.mean())
    return float((np.sum(np.abs((s * d).mean(1)) >= obs - 1e-12) + 1) / (R + 1))

def mcnemar_exact(a, b):
    from scipy.stats import binomtest
    a, b = np.asarray(a), np.asarray(b)
    n10, n01 = int(np.sum((a == 1) & (b == 0))), int(np.sum((a == 0) & (b == 1)))
    return n10, n01, (1.0 if n10 + n01 == 0 else float(binomtest(n10, n10 + n01, 0.5).pvalue))

def run_analysis(save=True):
    w = world()
    data = {c: load_rows(c) for c in ["B", "Bprime_v3", "Base"]}
    pv = {}   # (cond, variant) -> {vid: metrics}
    for c, rows in data.items():
        for (vid, var), r in rows.items():
            pv.setdefault((c, var), {})[vid] = per_video(r)

    ids_both = sorted(set.intersection(*[set(pv.get((c, "trajectory"), {})) for c in ("B", "Bprime_v3")]) &
                      set.intersection(*[set(pv.get((c, "final"), {})) for c in ("B", "Bprime_v3")]), key=nid)
    n = len(ids_both)
    log("=" * 78); log(f"CV ANALYSIS: {n}/50 videos have B and B'v3 on both prompts"); log("=" * 78)
    if n == 0:
        log("Nothing to analyse yet."); return
    if n < 50:
        log("  (partial run: numbers below are provisional)")

    # reference baselines computed from ground truth on the same videos
    gts = [w["by_id"][v] for v in ids_both]
    stay_rnd = np.mean([np.mean([p == g["initial_position"] for p in g["round_positions"]]) for g in gts])
    stay_fin = np.mean([g["final_position"] == g["initial_position"] for g in gts])
    maj = Counter(g["final_position"] for g in gts).most_common(1)[0][1] / n
    log(f"\nReference points on these {n} videos:  random 0.25 | majority-final {maj:.2f} | "
        f"stay-put per-round {stay_rnd:.2f}, stay-put final {stay_fin:.2f}")

    metrics = [("trajectory", "per_round", "per-round acc (trajectory prompt)  [primary]"),
               ("trajectory", "final",     "final acc (trajectory prompt)"),
               ("final",      "final",     "final acc (final-answer prompt)"),
               ("trajectory", "exact",     "exact 6-round match")]

    def vec(c, var, m, ids):
        return np.array([pv[(c, var)][v][m] for v in ids], float)

    log("\nPER-CONDITION RESULTS (mean over videos, 95% CI: Clopper-Pearson for binary, bootstrap for per-round)")
    log("-" * 78)
    out = {"n_videos": n, "ids": ids_both, "conditions": {}, "paired": {}, "per_fold": {}}
    for c in ("B", "Bprime_v3", "Base"):
        ids_c = [v for v in ids_both if v in pv.get((c, "trajectory"), {}) and v in pv.get((c, "final"), {})]
        if not ids_c:
            continue
        log(f"  {c}  (n={len(ids_c)})")
        out["conditions"][c] = {}
        for var, m, label in metrics:
            x = vec(c, var, m, ids_c)
            if m == "per_round":
                lo, hi = boot_ci(x)
            else:
                lo, hi = cp_ci(int(x.sum()), len(x))
            log(f"     {label:44s} {x.mean():.3f}  [{lo:.3f}, {hi:.3f}]")
            out["conditions"][c][f"{var}:{m}"] = {"mean": float(x.mean()), "ci": [float(lo), float(hi)], "n": len(x)}

    log("\nPAIRED COMPARISONS (video-level; diff = first minus second)")
    log("-" * 78)
    pairs = [("B", "Bprime_v3"), ("B", "Base"), ("Bprime_v3", "Base")]
    for a, b in pairs:
        ids_ab = [v for v in ids_both if all(v in pv.get((c, var), {}) for c in (a, b) for var in ("trajectory", "final"))]
        if len(ids_ab) < 2:
            continue
        log(f"  {a} vs {b}  (n={len(ids_ab)})")
        for var, m, label in metrics:
            xa, xb = vec(a, var, m, ids_ab), vec(b, var, m, ids_ab)
            d = xa - xb
            lo, hi = boot_ci(d)
            p_perm = signflip_p(d)
            extra = ""
            rec = {"mean_diff": float(d.mean()), "boot_ci": [float(lo), float(hi)], "perm_p": p_perm}
            if m != "per_round":
                n10, n01, pm = mcnemar_exact(xa, xb)
                extra = f" | McNemar exact {n10}:{n01} p={pm:.3f}"
                rec.update(discordant_a_only=n10, discordant_b_only=n01, mcnemar_p=pm)
            log(f"     {label:44s} diff {d.mean():+.3f} [{lo:+.3f}, {hi:+.3f}] | sign-flip p={p_perm:.3f}{extra}")
            out["paired"][f"{a}_vs_{b}::{var}:{m}"] = rec

    log("\nPER-FOLD (trajectory prompt, per-round acc): consistency check")
    log("-" * 78)
    log(f"  {'fold':5s} {'n':>3s} {'B':>7s} {'Bprime_v3':>10s} {'diff':>7s}   {'B final':>8s} {'Bp final':>9s}")
    for k in range(N_FOLDS):
        ids_k = [v for v in w["folds"][k] if v in ids_both]
        if not ids_k:
            continue
        b, bp = vec("B", "trajectory", "per_round", ids_k).mean(), vec("Bprime_v3", "trajectory", "per_round", ids_k).mean()
        bf, bpf = vec("B", "trajectory", "final", ids_k).mean(), vec("Bprime_v3", "trajectory", "final", ids_k).mean()
        log(f"  {k:<5d} {len(ids_k):>3d} {b:>7.3f} {bp:>10.3f} {b - bp:>+7.3f}   {bf:>8.2f} {bpf:>9.2f}")
        out["per_fold"][k] = {"n": len(ids_k), "B_per_round": float(b), "Bp_per_round": float(bp)}

    fails = sum(1 for c in ("B", "Bprime_v3", "Base") for r in data[c].values() if r["parsed"]["final"] is None)
    log(f"\nUnparsed outputs (counted as wrong): {fails}")
    log("\nHow to read this: video-level paired tests, one training run per fold (no seed repeats).")
    log("The intervals describe the recipe averaged over 5 different training sets, not seed-to-seed variance.")

    if save:
        os.makedirs(f"{CV_ROOT}/results", exist_ok=True)
        jdump(f"{CV_ROOT}/results/cv_summary.json", out)
        import csv
        with open(f"{CV_ROOT}/results/per_video.csv", "w", newline="") as f:
            wr = csv.writer(f)
            wr.writerow(["id", "fold", "cond", "variant", "final", "per_round", "exact"])
            for (c, var), d in pv.items():
                for v, m in sorted(d.items()):
                    wr.writerow([v, w["fold_of"][v], c, var, m["final"], round(m["per_round"], 4), m["exact"]])
        log(f"\nSaved: {CV_ROOT}/results/cv_summary.json and per_video.csv")

run_analysis()


CV ANALYSIS: 50/50 videos have B and B'v3 on both prompts

Reference points on these 50 videos:  random 0.25 | majority-final 0.30 | stay-put per-round 0.35, stay-put final 0.28

PER-CONDITION RESULTS (mean over videos, 95% CI: Clopper-Pearson for binary, bootstrap for per-round)
------------------------------------------------------------------------------
  B  (n=50)
     per-round acc (trajectory prompt)  [primary] 0.350  [0.287, 0.413]
     final acc (trajectory prompt)                0.240  [0.131, 0.382]
     final acc (final-answer prompt)              0.260  [0.146, 0.403]
     exact 6-round match                          0.000  [0.000, 0.071]
  Bprime_v3  (n=50)
     per-round acc (trajectory prompt)  [primary] 0.323  [0.260, 0.387]
     final acc (trajectory prompt)                0.260  [0.146, 0.403]
     final acc (final-answer prompt)              0.240  [0.131, 0.382]
     exact 6-round match                          0.000  [0.000, 0.071]
  Base  (n=50)
     per-round ac

In [6]:
for c, rows in {c: load_rows(c) for c in ["B", "Bprime_v3", "Base"]}.items():
    n = stay = const = copy_init = tot = 0
    s_ok = s_n = m_ok = m_n = 0
    for (vid, var), r in rows.items():
        if var != "trajectory":
            continue
        gt = r["ground_truth"]
        rd = {int(a): b for a, b in r["parsed"]["rounds"].items()}
        pred = [rd.get(i) for i in range(1, len(gt["round_positions"]) + 1)]
        init = gt["initial_position"]
        n += 1
        stay += all(p == init for p in pred)
        const += len(set(pred)) == 1
        for p, g in zip(pred, gt["round_positions"]):
            tot += 1
            copy_init += (p == init)
            if g == init:
                s_n += 1; s_ok += (p == g)
            else:
                m_n += 1; m_ok += (p == g)
    print(f"{c:10s} all-stay-put {stay}/{n} | constant seq {const}/{n} | pred==initial {copy_init/tot:.2f} | "
          f"acc when target at start {s_ok/max(s_n,1):.2f} (n={s_n}) | acc when moved away {m_ok/max(m_n,1):.2f} (n={m_n})")

B          all-stay-put 42/50 | constant seq 42/50 | pred==initial 0.93 | acc when target at start 0.93 (n=106) | acc when moved away 0.03 (n=194)
Bprime_v3  all-stay-put 32/50 | constant seq 33/50 | pred==initial 0.78 | acc when target at start 0.78 (n=106) | acc when moved away 0.07 (n=194)
Base       all-stay-put 0/50 | constant seq 0/50 | pred==initial 0.28 | acc when target at start 0.44 (n=106) | acc when moved away 0.07 (n=194)


In [7]:
def moved_acc(rows):
    out = {}
    for (vid, var), r in rows.items():
        if var != "trajectory":
            continue
        gt = r["ground_truth"]
        rd = {int(a): b for a, b in r["parsed"]["rounds"].items()}
        init = gt["initial_position"]
        hits = [rd.get(i) == g for i, g in enumerate(gt["round_positions"], 1) if g != init]
        if hits:
            out[vid] = (sum(hits), len(hits))
    return out

R = {c: moved_acc(load_rows(c)) for c in ["B", "Bprime_v3", "Base"]}
print("Pooled over rounds (CI treats rounds as independent, so optimistic):")
for c, d in R.items():
    x, n = sum(a for a, _ in d.values()), sum(b for _, b in d.values())
    lo, hi = cp_ci(x, n)
    print(f"  {c:10s} moved-away rounds correct: {x}/{n} = {x/n:.3f}  [{lo:.3f}, {hi:.3f}]")

print("\nVideo-level paired (cite these):")
for a, b in [("B", "Bprime_v3"), ("B", "Base"), ("Bprime_v3", "Base")]:
    ids = sorted(set(R[a]) & set(R[b]))
    d = np.array([R[a][v][0] / R[a][v][1] - R[b][v][0] / R[b][v][1] for v in ids])
    lo, hi = boot_ci(d)
    print(f"  {a} vs {b}: diff {d.mean():+.3f} [{lo:+.3f}, {hi:+.3f}] sign-flip p={signflip_p(d):.3f} (n={len(ids)} videos)")

Pooled over rounds (CI treats rounds as independent, so optimistic):
  B          moved-away rounds correct: 6/194 = 0.031  [0.011, 0.066]
  Bprime_v3  moved-away rounds correct: 14/194 = 0.072  [0.040, 0.118]
  Base       moved-away rounds correct: 13/194 = 0.067  [0.036, 0.112]

Video-level paired (cite these):
  B vs Bprime_v3: diff -0.045 [-0.089, -0.011] sign-flip p=0.031 (n=50 videos)
  B vs Base: diff -0.021 [-0.074, +0.035] sign-flip p=0.516 (n=50 videos)
  Bprime_v3 vs Base: diff +0.023 [-0.036, +0.087] sign-flip p=0.492 (n=50 videos)
